In [1]:
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

In [2]:
import torch
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification,
    TrainingArguments, 
    Trainer
)
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

/opt/anaconda3/envs/pytorch/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# device
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Use: {device}")

# load model and tokenizer
model_name = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"

tokenizer = AutoTokenizer.from_pretrained(model_name)
# padding token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# classification model(negative, neutral, positive）
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3,
    id2label={0: "negative", 1: "neutral", 2: "positive"},
    label2id={"negative": 0, "neutral": 1, "positive": 2},
    torch_dtype=torch.float32
)

model.to(device)

Use: mps


`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 338/338 [00:04<00:00, 76.90it/s] 
Qwen2ForSequenceClassification LOAD REPORT from: deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B
Key            | Status     | 
---------------+------------+-
lm_head.weight | UNEXPECTED | 
score.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Qwen2ForSequenceClassification(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((1536,), eps=1e-

In [4]:
import datasets

# load dataset
def load_financial_phrasebank(filepath):
    """
    load financial_phrasebank dataset
    Returns:
        datasets.Dataset
    """
    sentences = []
    labels = []
    
    # format: sentence@label
    with open(filepath, encoding="iso-8859-1") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            sentence, label = line.rsplit("@", 1)
            sentences.append(sentence)
            labels.append(label.strip())
    
    # to Hugging Face Dataset format
    dataset = datasets.Dataset.from_dict({
        "sentence": sentences,
        "label": labels,
    })
    
    label_mapping = {"negative": 0, "neutral": 1, "positive": 2}
    dataset = dataset.map(lambda x: {"label": label_mapping[x["label"]]})
    
    return dataset

dataset = load_financial_phrasebank("FinancialPhraseBank-v1.0/Sentences_AllAgree.txt")

# split dataset
train_test_split = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = train_test_split["train"]
eval_dataset = train_test_split["test"]

print(f"Training set size: {len(train_dataset)}")
print(f"Test set size: {len(eval_dataset)}")

Map: 100%|██████████| 2264/2264 [00:00<00:00, 65994.65 examples/s]

Training set size: 2037
Test set size: 227


In [5]:
# preprocess
def preprocess_function(examples):
    """tokenizing"""
    return tokenizer(
        examples["sentence"],
        truncation=True,
        padding="max_length",
        max_length=256,
        return_tensors=None
    )

tokenized_train = train_dataset.map(preprocess_function, batched=True)
tokenized_eval = eval_dataset.map(preprocess_function, batched=True)

tokenized_train = tokenized_train.rename_column("label", "labels")
tokenized_eval = tokenized_eval.rename_column("label", "labels")
tokenized_train.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
tokenized_eval.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

tokenized_train = tokenized_train.map(lambda x: {"labels": int(x["labels"])})
tokenized_eval = tokenized_eval.map(lambda x: {"labels": int(x["labels"])})

Map: 100%|██████████| 227/227 [00:00<00:00, 12464.25 examples/s]


In [ ]:
# LoRA setting
lora_config = LoraConfig(
    r=8,                      # LoRA rank
    lora_alpha=32,            # scaling param
    target_modules=["q_proj", "v_proj"],  # focus on attention layer
    lora_dropout=0.1,         # Dropout prob
    bias="none",
    task_type=TaskType.SEQ_CLS  # sequencial classification
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


trainable params: 1,094,144 || all params: 1,544,813,056 || trainable%: 0.0708


In [ ]:
# training params
training_args = TrainingArguments(
    output_dir="./fin_deepseek_lora_mac",
    num_train_epochs=5,                    
    per_device_train_batch_size=2,         
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,         # batch size = 2*4=8
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_ratio=0.1,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=False,                            
    report_to="none",                      
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [8]:
# evaluation
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    accuracy = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average="weighted")
    return {"accuracy": accuracy, "f1": f1}

In [9]:
if model.config.pad_token_id is None:
    model.config.pad_token_id = tokenizer.eos_token_id

# training
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    # tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

print("finetuning started...")
trainer.train()

finetuning started...


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.545530,0.204327,0.947137,0.947682
2,0.231857,0.231482,0.947137,0.947262
3,0.052452,0.176029,0.964758,0.964481
4,0.000751,0.192243,0.960352,0.959933
5,0.000080,0.192508,0.964758,0.964366


TrainOutput(global_step=1275, training_loss=0.5738089116446345, metrics={'train_runtime': 29367.5478, 'train_samples_per_second': 0.347, 'train_steps_per_second': 0.043, 'total_flos': 2.05163671781376e+16, 'train_loss': 0.5738089116446345, 'epoch': 5.0})

In [10]:
# save model
model.save_pretrained("./fin_deepseek_lora_mac/final")
tokenizer.save_pretrained("./fin_deepseek_lora_mac/final")
print("model saved")

model saved


In [ ]:
# test
from peft import PeftModel

base_model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3,
    id2label={0: "negative", 1: "neutral", 2: "positive"},
    label2id={"negative": 0, "neutral": 1, "positive": 2}
)
loaded_model = PeftModel.from_pretrained(base_model, "./fin_deepseek_lora_mac/final")
loaded_model.to(device)

test_text = "The company's quarterly earnings exceeded all market expectations."
inputs = tokenizer(test_text, return_tensors="pt", truncation=True, max_length=256).to(device)
with torch.no_grad():
    outputs = loaded_model(**inputs)
predicted_class = torch.argmax(outputs.logits, dim=-1).item()
print(f"news: {test_text}")
print(f"predicted tone: {loaded_model.config.id2label[predicted_class]}")

Loading weights: 100%|██████████| 338/338 [00:00<00:00, 7367.99it/s]
Qwen2ForSequenceClassification LOAD REPORT from: deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B
Key            | Status     | 
---------------+------------+-
lm_head.weight | UNEXPECTED | 
score.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


news: The company's quarterly earnings exceeded all market expectations.
predicted tone: positive
